# Divide Two Integers

**Problem Link:**  
https://leetcode.com/problems/divide-two-integers/description/

## Problem Statement
Given two integers `dividend` and `divisor`, divide two integers **without** using multiplication, division, or the mod operator.

Integer division should truncate toward zero (e.g. `8.345` → `8`, `-2.7335` → `-2`).

Return the quotient. If the quotient is outside the 32-bit signed integer range `[-2³¹, 2³¹ - 1]`, clamp it to the boundary.

## Examples

### Example 1
Input: `dividend = 10, divisor = 3`  
Output: `3`  
Explanation: `10 / 3 = 3.333...` truncated to `3`.

### Example 2
Input: `dividend = 7, divisor = -3`  
Output: `-2`  
Explanation: `7 / -3 = -2.333...` truncated to `-2`.

## Constraints
- `-2³¹ <= dividend, divisor <= 2³¹ - 1`
- `divisor != 0`


## Bit Shifting (Exponential Search)

### Strategy
We cannot use `*`, `/`, or `%`, but we **can** use bit shifts. The idea is to find the largest multiple of `divisor` that fits into `dividend` using powers of two:

- Work with absolute values to simplify the logic; track the sign separately
- In each outer loop iteration, find the largest `k` such that `divisor << k <= remaining dividend`
  - This is done by doubling (`<< 1`) the divisor until it would exceed the dividend
  - At that point, `divisor << k` is the largest fitting multiple — subtract it from the dividend and add `1 << k` to the quotient
- Repeat until the dividend is smaller than the original divisor
- Apply the sign and clamp to `[-2³¹, 2³¹ - 1]`

### Edge Case
- `-2³¹ / -1 = 2³¹` overflows the 32-bit range — handle explicitly by returning `INT_MAX`

### Time Complexity
- **O(log² n)** — the outer loop runs O(log n) times and the inner doubling loop also runs O(log n) times

### Space Complexity
- **O(1)** extra space


In [1]:
class Solution:
    def divide(self, dividend: int, divisor: int) -> int:
        INT_MIN = -(2 ** 31)
        INT_MAX =  (2 ** 31) - 1

        # Only overflow case: -2^31 / -1 = 2^31 which exceeds INT_MAX
        if dividend == INT_MIN and divisor == -1:
            return INT_MAX

        # Determine sign and work with absolute values
        negative = (dividend < 0) != (divisor < 0)
        a = abs(dividend)
        b = abs(divisor)

        quotient = 0

        while a >= b:
            temp  = b
            multiple = 1
            # Double temp until it would exceed a
            while a >= (temp << 1):
                temp     <<= 1
                multiple <<= 1
            a        -= temp
            quotient += multiple

        result = -quotient if negative else quotient
        return max(INT_MIN, min(INT_MAX, result))


In [2]:
def test_divide():
    sol = Solution()

    assert sol.divide(10,  3)   ==  3
    assert sol.divide(7,  -3)   == -2

    # Truncation toward zero
    assert sol.divide(-7,  2)   == -3
    assert sol.divide(-7, -2)   ==  3

    # Exact division
    assert sol.divide(12,  4)   ==  3

    # Divisor larger than dividend
    assert sol.divide(1,   2)   ==  0

    # Divide by 1
    assert sol.divide(100, 1)   == 100

    # Overflow edge case: -2^31 / -1 must clamp to INT_MAX
    assert sol.divide(-(2**31), -1) == (2**31) - 1

    # INT_MIN / 1
    assert sol.divide(-(2**31),  1) == -(2**31)

    # Same value
    assert sol.divide(5,   5)   ==  1
    assert sol.divide(-5, -5)   ==  1

    print("All test cases passed!")

test_divide()


All test cases passed!
